# CMIP6 metadata and clisops subsetting

Compare synthetic NetCDF controls and metadata variants using the public clisops subset operator. Each case checks xarray decoding first, then runs independent time-only and geographic bounding-box requests. The checks compare the returned coordinates and data values with explicit selections.

In [ ]:
import sys
from importlib.metadata import version

import ipytest
import numpy as np
import pytest
import xarray as xr
from clisops.ops.subset import subset
from netCDF4 import Dataset

ipytest.autoconfig(raise_on_error=True, magics=False, addopts=())

print(f"Python {sys.version.split()[0]}")
for package in ("clisops", "xarray", "cf-xarray", "numpy", "netCDF4", "cftime", "pytest", "ipytest"):
    print(f"{package} {version(package)}")

## Shared test data

Use four daily samples from 1 to 4 January 1850 and request the middle two days. The bbox is 12°W to 3°W and 43°N to 52°N. On the rectilinear grid it includes latitudes 45° and 50° and longitudes 10°W and 5°W. Unique data values make a shifted selection visible.

In [ ]:
TIME_UNITS = "days since 1850-01-01 00:00:00"
DATES = np.arange("1850-01-01", "1850-01-05", dtype="datetime64[D]").astype("datetime64[ns]")
TIME_RANGE = "1850-01-02T00:00:00/1850-01-03T00:00:00"
BBOX = (-12, 43, -3, 52)  # west, south, east, north

time = (
    "time",
    np.arange(4),
    {"units": TIME_UNITS, "calendar": "standard", "axis": "T", "standard_name": "time"},
)
latitude = (
    "lat",
    [40, 45, 50, 55],
    {"units": "degrees_north", "axis": "Y", "standard_name": "latitude"},
)
longitude = (
    "lon",
    [-15, -10, -5, 0, 5],
    {"units": "degrees_east", "axis": "X", "standard_name": "longitude"},
)
pressure = (
    "plev",
    [100000, 85000],
    {"units": "Pa", "axis": "Z", "standard_name": "air_pressure", "positive": "down"},
)

rectilinear = xr.Dataset(
    {"ta": (("time", "plev", "lat", "lon"), np.arange(160.0).reshape(4, 2, 4, 5))},
    coords={"time": time, "plev": pressure, "lat": latitude, "lon": longitude},
)
rectilinear_bbox = {"lat": slice(1, 3), "lon": slice(1, 3)}

In [ ]:
cases = []


def add_case(name, raw, variable, bbox_index, bbox_mask=None):
    # Use positional expectations, independently of clisops' coordinate discovery.
    expected = raw[variable].assign_coords(time=DATES)
    expected_bbox = expected.isel(bbox_index)
    if bbox_mask is not None:
        expected_bbox = expected_bbox.where(bbox_mask)
    cases.append(pytest.param((raw, expected, expected_bbox), id=name))

## 1. Time units

Keep the numeric offsets and calendar fixed. Change only the formatting of the reference time in the units string.

In [ ]:
add_case("rectilinear-control", rectilinear, "ta", rectilinear_bbox)

short_time_units = rectilinear.copy(deep=True)
short_time_units.time.attrs["units"] = "days since 1850-01-01 0:0:0.0"
add_case("unpadded-time-units", short_time_units, "ta", rectilinear_bbox)

## 2. Missing coordinate metadata

Use the rectilinear control from case 1. Remove one attribute from one coordinate at a time. Both pressure levels should be preserved by each request.

In [ ]:
for coordinate in ("time", "lat", "lon", "plev"):
    for attribute in ("axis", "standard_name"):
        missing_attribute = rectilinear.copy(deep=True)
        del missing_attribute[coordinate].attrs[attribute]
        add_case(f"{coordinate}-without-{attribute}", missing_attribute, "ta", rectilinear_bbox)

## 3. Degree-valued projected axes

Keep the 2-D geographic coordinates identical in both fixtures. Replace the metre-valued x/y axes with small degree-valued axes, retaining their projected standard names. The geographic bbox retains three cells in a 2 × 2 window. The northeast cell is outside the bbox and should be masked.

On the metre-valued control, clisops keeps the 2 × 2 window and its latitude and longitude coordinates. It masks the northeast `tas` value with `NaN` at every selected time. The expected mask applies only to the data values.

In [ ]:
curvilinear = xr.Dataset(
    {"tas": (("time", "y", "x"), np.arange(48.0).reshape(4, 3, 4))},
    coords={
        "time": time,
        "x": ("x", [0, 500_000, 1_000_000, 1_500_000],
              {"standard_name": "projection_x_coordinate", "units": "m", "axis": "X"}),
        "y": ("y", [0, 500_000, 1_000_000],
              {"standard_name": "projection_y_coordinate", "units": "m", "axis": "Y"}),
        "lat": (("y", "x"), [[39, 40, 41, 42], [43, 45, 46, 47], [49, 50, 53, 54]],
                {"standard_name": "latitude", "units": "degrees_north"}),
        "lon": (("y", "x"), [[-16, -11, -6, -1], [-15, -10, -5, 0], [-14, -9, -4, 1]],
                {"standard_name": "longitude", "units": "degrees_east"}),
    },
)
curvilinear_bbox = {"y": slice(1, 3), "x": slice(1, 3)}
curvilinear_mask = xr.DataArray([[True, True], [True, False]], dims=("y", "x"))

add_case("projected-axes-in-metres", curvilinear, "tas", curvilinear_bbox, curvilinear_mask)

axes_in_degrees = curvilinear.assign_coords(
    x=("x", [0, 5, 10, 15], {**curvilinear.x.attrs, "units": "degrees"}),
    y=("y", [0, 5, 10], {**curvilinear.y.attrs, "units": "degrees"}),
)
add_case("projected-axes-in-degrees", axes_in_degrees, "tas", curvilinear_bbox, curvilinear_mask)

## 4. Missing auxiliary height coordinate

The control has a scalar 2 m height. Both files declare `coordinates="height"` on `tas`. The variant omits the height variable. Xarray preserves the dangling reference in `tas.encoding["coordinates"]` after CF decoding.

In [ ]:
height_control = xr.Dataset(
    {"tas": (("time", "lat", "lon"), np.arange(80.0).reshape(4, 4, 5), {"coordinates": "height"})},
    coords={
        "time": time,
        "lat": latitude,
        "lon": longitude,
        "height": ((), 2.0, {"standard_name": "height", "units": "m", "positive": "up"}),
    },
)
add_case("height-control", height_control, "tas", rectilinear_bbox)
missing_height = height_control.drop_vars("height")
add_case("missing-height", missing_height, "tas", rectilinear_bbox)

## Pytest fixtures and tests

Each case is written to a temporary NetCDF file and opened with xarray. Pytest collects four tests per case for decoding, time subsetting, bbox subsetting, and combined time and bbox subsetting. A negative control removes the 2-D latitude and longitude coordinates from the degree-valued projected grid and checks the exception from the bbox request.

In [ ]:
ipytest.clean()


@pytest.fixture(scope="module", params=cases)
def case(request):
    return request.param


@pytest.fixture(scope="module")
def decoded(case, tmp_path_factory):
    raw, expected, _ = case
    path = tmp_path_factory.mktemp("cmip6") / "fixture.nc"
    raw.to_netcdf(path, engine="netcdf4")

    # Confirm that xarray wrote the exact strings the decoder should see.
    with Dataset(path) as file:
        assert file["time"].units == raw["time"].attrs["units"]
        if raw is height_control or raw is missing_height:
            assert file[expected.name].coordinates == "height"
            assert ("height" in file.variables) == (raw is height_control)

    with xr.open_dataset(path, engine="netcdf4") as dataset:
        return dataset.load()


def test_xarray_open(case, decoded):
    raw, expected, _ = case
    assert decoded[expected.name].equals(expected)
    if raw is missing_height:
        assert decoded[expected.name].encoding.get("coordinates") == "height"
        assert "height" not in decoded.variables


def test_time_subset(case, decoded):
    _, expected, _ = case
    outputs = subset(decoded.copy(deep=True), time=TIME_RANGE, output_type="xarray")
    assert len(outputs) == 1
    assert outputs[0][expected.name].equals(expected.isel(time=slice(1, 3)))


def test_bbox_subset(case, decoded):
    _, _, expected_bbox = case
    outputs = subset(decoded.copy(deep=True), area=BBOX, output_type="xarray")
    assert len(outputs) == 1
    assert outputs[0][expected_bbox.name].equals(expected_bbox)


def test_time_and_bbox_subset(case, decoded):
    _, _, expected_bbox = case
    outputs = subset(decoded.copy(deep=True), time=TIME_RANGE, area=BBOX, output_type="xarray")
    assert len(outputs) == 1
    assert outputs[0][expected_bbox.name].equals(expected_bbox.isel(time=slice(1, 3)))


def test_bbox_without_geographic_coordinates():
    without_lat_lon = xr.decode_cf(axes_in_degrees.drop_vars(["lat", "lon"]))
    with pytest.raises(AttributeError, match="has no attribute 'lon'") as error:
        subset(without_lat_lon, area=BBOX, output_type="xarray")

    print(f"Bbox without geographic coordinates raised {error.type.__name__}: {error.value}")

## Run pytest

In [ ]:
ipytest.run("-v", "-rP", "--color=no", "-p", "no:cacheprovider")